# Agent 2: People/Transport (Phase 4)

This notebook subscribes to weather updates and publishes derived people state over MQTT.

In [ ]:
# Cell purpose: Load configuration and define weather input + people output topics.
from __future__ import annotations

import json
import time
from datetime import UTC, datetime

from simulated_city.config import load_config
from simulated_city import mqtt

cfg = load_config()
sim_cfg = cfg.simulation
if sim_cfg is None:
    raise ValueError("Missing simulation config. Add simulation.weather_agent in config.yaml.")

weather_topic = f"{cfg.mqtt.base_topic}/{sim_cfg.weather_agent.topic_suffix}"
people_topic = f"{cfg.mqtt.base_topic}/people/state"

print(f"Subscribe topic: {weather_topic}")
print(f"Publish topic: {people_topic}")

In [ ]:
# Cell purpose: Connect to MQTT, subscribe to weather, and cache the latest weather state.
latest_weather: dict[str, object] | None = None

def on_message(client, userdata, msg):
    global latest_weather
    payload = json.loads(msg.payload.decode())
    latest_weather = payload
    print(f"Received weather tick={payload.get('tick')} condition={payload.get('condition')}")

client = mqtt.connect_mqtt(cfg.mqtt, client_id_suffix="agent-people")
client.on_message = on_message
client.subscribe(weather_topic, qos=1)
print("People agent connected and subscribed.")

In [ ]:
# Cell purpose: Publish derived people state from subscribed weather data.
# Run this while weather agent publishes. Stop with KeyboardInterrupt when done.
try:
    while True:
        if latest_weather is not None:
            condition = str(latest_weather.get("condition", "clear"))
            raining = condition == "rain"

            people_state = {
                "agent": "people",
                "timestamp": datetime.now(UTC).isoformat(),
                "source_weather_tick": latest_weather.get("tick"),
                "source_condition": condition,
                "movement_mode": "to_coffee_shops" if raining else "random_walk",
                "active_people": 5,
            }

            mqtt.publish_json_checked(client, people_topic, people_state)
            print(f"Published people state from weather tick={people_state['source_weather_tick']}")
            latest_weather = None

        time.sleep(0.2)
except KeyboardInterrupt:
    print("Stopping people agent loop.")
finally:
    client._simcity_connector.disconnect()